# HYSPLIT Quick Start

This notebook demonstrates the core functionality of the `hysplit` package.

**Requirements:**
- Internet connection (to download meteorological data)
- HYSPLIT binary (included in package)

## Setup

In [ ]:
# Add parent directory to path for local development
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import hysplit
print(f"hysplit version: {hysplit.__version__}")

# Create directories for meteorological data and output
met_dir = Path("./met")
out_dir = Path("./out")
met_dir.mkdir(exist_ok=True)
out_dir.mkdir(exist_ok=True)
print(f"Met directory: {met_dir.absolute()}")
print(f"Output directory: {out_dir.absolute()}")

## 1. Run a Forward Trajectory

Run a 24-hour forward trajectory from Ontario, Canada:

In [ ]:
trajectory = hysplit.hysplit_trajectory(
    lat=42.83752,              # Starting latitude
    lon=-80.30364,             # Starting longitude
    height=50,                 # Starting height (m AGL)
    duration=24,               # Duration (hours)
    days=["2012-03-12"],       # Date(s) to run
    daily_hours=[0, 6, 12, 18],# Hours to start trajectories
    direction="forward",       # Forward in time
    met_type="reanalysis",     # Meteorological data type
    met_dir=str(met_dir),      # Where to store met data
    exec_dir=str(out_dir)      # Working directory
)

print(f"Trajectory shape: {trajectory.shape}")
print(f"Columns: {list(trajectory.columns)}")

In [ ]:
# View the first 10 rows
trajectory.head(10)

## 2. Trajectory Summary

In [ ]:
print("Trajectory Summary:")
print(f"  Number of runs: {trajectory['run'].nunique()}")
print(f"  Points per run: {len(trajectory) // trajectory['run'].nunique()}")
print(f"  Lat range: {trajectory['lat'].min():.4f} to {trajectory['lat'].max():.4f}")
print(f"  Lon range: {trajectory['lon'].min():.4f} to {trajectory['lon'].max():.4f}")
print(f"  Height range: {trajectory['height'].min():.1f} to {trajectory['height'].max():.1f} m")

## 3. Plot Trajectories

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Trajectory paths
ax = axes[0]
for run_id in trajectory['run'].unique():
    run_data = trajectory[trajectory['run'] == run_id]
    ax.plot(run_data['lon'], run_data['lat'], 'o-', markersize=3, alpha=0.7)

# Mark starting point
ax.scatter(-80.30364, 42.83752, c='red', s=150, marker='*', zorder=10, label='Start')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Trajectory Paths')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Height profile
ax = axes[1]
for run_id in trajectory['run'].unique():
    run_data = trajectory[trajectory['run'] == run_id]
    ax.plot(run_data['hour_along'], run_data['height'], 'o-', markersize=3, alpha=0.7)

ax.set_xlabel('Hours Along Trajectory')
ax.set_ylabel('Height (m)')
ax.set_title('Height Profile')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Backward Trajectory

Run a backward trajectory to find where air parcels came from:

In [ ]:
backward_traj = hysplit.hysplit_trajectory(
    lat=43.65,                 # Toronto
    lon=-79.38,
    height=100,
    duration=48,               # 48 hours back
    days=["2012-03-12"],
    daily_hours=[12],          # Noon
    direction="backward",      # Backward in time
    met_type="reanalysis",
    met_dir=str(met_dir),
    exec_dir=str(out_dir)
)

print(f"Backward trajectory: {len(backward_traj)} points")
print(f"Starting location: ({backward_traj['lat'].iloc[0]:.2f}, {backward_traj['lon'].iloc[0]:.2f})")
print(f"Origin (48h ago): ({backward_traj['lat'].iloc[-1]:.2f}, {backward_traj['lon'].iloc[-1]:.2f})")

## 5. Method Chaining API

For more control, use the object-oriented interface:

In [ ]:
model = (
    hysplit.create_trajectory_model()
    .add_trajectory_params(
        lat=49.28,              # Vancouver
        lon=-123.12,
        height=50,
        duration=24,
        days=["2012-03-12"],
        daily_hours=[0, 12],
        direction="forward",
        met_type="reanalysis",
        met_dir=str(met_dir),
        exec_dir=str(out_dir)
    )
    .run()
)

result = model.get_output()
print(f"Vancouver trajectories: {result.shape}")
result.head()

## Done!

You've successfully run HYSPLIT trajectories using Python. Key functions:

- `hysplit.hysplit_trajectory()` - Simple function interface
- `hysplit.create_trajectory_model()` - Method chaining interface

**Meteorological data types:**
- `reanalysis` - NCEP/NCAR Reanalysis (2.5°, global, 1948-present)
- `gdas1` - GDAS (1°, global, recent)
- `gdas0.5` - GDAS (0.5°, global, recent)
- `nam12` - NAM (12km, North America)
- `gfs0.25` - GFS (0.25°, global)